# Fit mô hình cho demo: 9 loại × 3 phương pháp (Google Colab)

Fit Autoencoder, PaDiM, PatchCore cho 9 loại của thí nghiệm TN1–TN5, cùng cấu hình với `src/tn`:

| Bộ | Loại | Kích thước | Coreset | Ngưỡng |
|---|---|---|---|---|
| MVTec AD | grid, screw, transistor | 256×256 | 10% | 15% train/good tách riêng |
| MVTec AD 2 | vial, sheet_metal | 256×192, 256×1024 | 1% | validation/good |
| VisA | pcb1–pcb4 | 256×320 | 1% | 15% train tách riêng |

Ngưỡng = trung bình + 3 độ lệch chuẩn trên ảnh good validation.

Code đã nhúng sẵn trong notebook. Chọn *Runtime → Change runtime type → T4 GPU*, rồi *Run all*.
Kết quả ghi thẳng lên `MyDrive/goi_colab/checkpoints_tn/<loại>/`. Colab ngắt giữa chừng thì chạy lại: loại đã xong được bỏ qua.

## 1. Drive, code, GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import base64, io, sys, zipfile
from pathlib import Path
PKG = Path('/content/drive/MyDrive/goi_colab')
PKG.mkdir(parents=True, exist_ok=True)
CODE_ZIP = '...'
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP))).extractall('/content/repo')   # src/tn + tools/build_tn_models.py
sys.path.insert(0, '/content/repo')

from src.tn.config import require_gpu, check_gpu
check_gpu(); require_gpu()

## 2. Tải dữ liệu (~15 phút)
Tải về ổ local của Colab: MVTec AD 3 loại từ bản sao Hugging Face (~730 MB; link tar.xz cũ của MVTec đã chết), MVTec AD 2 vial + sheet_metal từ trang MVTec (~2,5 GB), VisA từ AWS (~1,8 GB).

In [ ]:
from src.tn import data as D
DATA = Path('/content/datasets')
!python /content/repo/tools/download_mvtec.py {DATA}/MVTecAD grid screw transistor
D.prepare_ad2(DATA / 'MVTecAD2', ['vial', 'sheet_metal'])
D.prepare_visa(DATA / 'VisA', ['pcb1', 'pcb2', 'pcb3', 'pcb4'])

## 3. Fit 9 loại (~40–60 phút trên T4)

In [ ]:
OUT = PKG / 'checkpoints_tn'
!python /content/repo/tools/build_tn_models.py --data {DATA} --out {OUT}

## 4. Kiểm tra kết quả

In [ ]:
import json, pandas as pd
rows = []
for d in sorted(OUT.iterdir()):
    if (d / 'meta.json').exists():
        m = json.loads((d / 'meta.json').read_text(encoding='utf-8'))
        for meth, met in m['metrics'].items():
            rows.append(dict(loai=d.name, pp=meth, **met))
    else:
        print('CHƯA XONG:', d.name)
pd.DataFrame(rows).pivot_table(index='loai', columns='pp', values=['i_auroc', 'aupro', 'img_fpr_thr']).round(3)

In [ ]:
!du -sh {OUT}/*